# 2. 토큰과 과금

**목표** — 토큰이 무엇인지 직접 세어보고, 내가 방금 한 호출이 **돈으로 얼마인지** 계산한다.

**소요 시간** 약 80분

| 다루는 것 | |
| --- | --- |
| 1 | 토큰 세어보기 |
| 2 | 한국어 vs 영어 — 어느 쪽이 비싼가 |
| 3 | 응답에서 사용량 읽기 |
| 4 | 단가를 찾아 비용 계산하기 |
| 5 | 서비스 규모로 환산 |
| 6 | 무료 한도와 `429` 체험 |
| 7 | 연습문제 |

> **이 노트북이 오늘의 핵심이다.** 여기서 만든 비용 계산 함수를 노트북 03, 04에서 계속 재사용한다.

## 0. 준비

In [15]:
import os
import time

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from openai import OpenAI

load_dotenv(override=True)

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
oa = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gemini-3.1-flash-lite"
OA_MODEL = "gpt-4o-mini"

print("준비 완료")

준비 완료


## 1. 토큰 세어보기

모델은 글자가 아니라 **토큰** 단위로 텍스트를 다룬다. `count_tokens()`는 **호출 없이(= 과금 없이)** 토큰 수만 알려준다.

먼저 예측해보자. 아래 문자열들은 각각 몇 토큰일까?

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
samples = [
    "안녕",
    "안녕하세요",
    "안녕하세요 반갑습니다",
    "Hello",
    "Hello, nice to meet you",
    "1234567890",
    "         ",
]

rows = []
for s in samples:
    n = client.models.count_tokens(model=MODEL, contents=s).total_tokens
    rows.append({"텍스트": repr(s), "글자수": len(s), "토큰수": n})

pd.DataFrame(rows)
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 글자 수와 토큰 수가 비례하지 않는지 본다

In [16]:
samples = [
    "안녕",
    "안녕하세요",
    "안녕하세요 반갑습니다",
    "Hello",
    "Hello, nice to meet you",
    "1234567890",
    "         ",
]

In [17]:
rows = []
for s in samples:
    n = client.models.count_tokens(model=MODEL, contents=s).total_tokens
    rows.append({"텍스트": repr(s), "글자수": len(s), "토큰수": n})

pd.DataFrame(rows)

,텍스트,글자수,토큰수
0,'안녕',2,3
1,'안녕하세요',5,2
2,'안녕하세요 반갑습니다',11,3
3,'Hello',5,2
4,"'Hello, nice to meet you'",23,7
5,'1234567890',10,11
6,' ',9,2


### 관찰 포인트

- **글자 수와 토큰 수는 비례하지 않는다.** 자주 쓰이는 표현일수록 적은 토큰으로 쪼개진다.
- 공백과 숫자도 토큰을 차지한다 — **프롬프트에 넣은 모든 것이 비용이다.**
- 토큰화 방식은 모델마다 다르다. 같은 문장도 Gemini와 OpenAI의 토큰 수가 다르다.

## 2. 한국어 vs 영어 — 어느 쪽이 비싼가

"한국어는 영어보다 토큰을 훨씬 많이 먹는다"는 말을 자주 듣는다. **직접 재서 확인해본다.**

**같은 의미**의 문장을 한국어와 영어로 각각 재고, 글자당 토큰 효율도 같이 본다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
pairs = [
    ("안녕하세요, 만나서 반갑습니다.", "Hello, nice to meet you."),
    ("오늘 날씨가 정말 좋네요.", "The weather is really nice today."),
    ("이 제품의 반품 정책을 알려주세요.", "Please tell me the return policy for this product."),
    ("인공지능은 컴퓨터가 사람처럼 학습하고 판단하는 기술이다.",
     "Artificial intelligence is a technology where computers learn and make decisions like humans."),
]

rows = []
for ko, en in pairs:
    ko_n = client.models.count_tokens(model=MODEL, contents=ko).total_tokens
    en_n = client.models.count_tokens(model=MODEL, contents=en).total_tokens
    rows.append({
        "한국어": ko[:20],
        "KO 토큰": ko_n,
        "EN 토큰": en_n,
        "배율": round(ko_n / en_n, 2),
        "KO 글자/토큰": round(len(ko) / ko_n, 2),
        "EN 글자/토큰": round(len(en) / en_n, 2),
    })

df = pd.DataFrame(rows)
total_ko = sum(r["KO 토큰"] for r in rows)
total_en = sum(r["EN 토큰"] for r in rows)
print(f"전체 배율: {total_ko / total_en:.2f}배  (KO {total_ko}토큰 / EN {total_en}토큰)")
df
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 배율 열에 1보다 작은 행이 있는지 본다

In [18]:
pairs = [
    ("안녕하세요, 만나서 반갑습니다.", "Hello, nice to meet you."),
    ("오늘 날씨가 정말 좋네요.", "The weather is really nice today."),
    ("이 제품의 반품 정책을 알려주세요.", "Please tell me the return policy for this product."),
    ("인공지능은 컴퓨터가 사람처럼 학습하고 판단하는 기술이다.",
     "Artificial intelligence is a technology where computers learn and make decisions like humans."),
]

In [19]:
rows = []
for ko, en in pairs:
    ko_n = client.models.count_tokens(model=MODEL, contents=ko).total_tokens
    en_n = client.models.count_tokens(model=MODEL, contents=en).total_tokens
    rows.append({
        "한국어": ko[:20],
        "KO 토큰": ko_n,
        "EN 토큰": en_n,
        "배율": round(ko_n / en_n, 2),
        "KO 글자/토큰": round(len(ko) / ko_n, 2),
        "EN 글자/토큰": round(len(en) / en_n, 2),
    })

In [20]:
df = pd.DataFrame(rows)
total_ko = sum(r["KO 토큰"] for r in rows)
total_en = sum(r["EN 토큰"] for r in rows)
print(f"전체 배율: {total_ko / total_en:.2f}배  (KO {total_ko}토큰 / EN {total_en}토큰)")
df

전체 배율: 1.07배  (KO 45토큰 / EN 42토큰)


,한국어,KO 토큰,EN 토큰,배율,KO 글자/토큰,EN 글자/토큰
0,"안녕하세요, 만나서 반갑습니다.",7,8,0.88,2.43,3.00
1,오늘 날씨가 정말 좋네요.,9,8,1.12,1.56,4.12
2,이 제품의 반품 정책을 알려주세요.,12,11,1.09,1.58,4.55
3,인공지능은 컴퓨터가 사람처럼 학습하고,17,15,1.13,1.82,6.20


### 결과를 어떻게 읽어야 하나

**두 가지 다른 이야기가 섞여 있다.**

| 기준 | 결과 | 의미 |
| --- | --- | --- |
| **글자당 토큰** | 한국어가 확실히 불리 (영어의 절반 수준) | 글자 하나를 표현하는 데 토큰을 더 쓴다 |
| **문장당 토큰** | 거의 비슷하거나 조금 불리 | 한국어는 같은 뜻을 **더 적은 글자로** 쓰기 때문에 상쇄된다 |

문장에 따라서는 **한국어가 오히려 적게 나오는 경우도 있다.** 위 표의 `배율` 열에서 1보다 작은 행이 있는지 확인해보자.

> 예전 모델에서는 한국어가 영어의 2~3배까지 나왔지만, **최신 토크나이저는 한국어를 훨씬 잘 처리한다.**
> 그래서 "한국어는 무조건 비싸다"는 통념을 그대로 믿으면 안 된다.

### 여기서 진짜 배울 것

**전해 들은 수치를 믿지 말고 직접 측정한다.** 토큰 수는 모델·토크나이저 버전마다 다르고, 그 차이가 그대로 비용이 된다.

토큰이 많아지면 세 가지가 함께 나빠진다.

1. **비용** — 토큰 단위 과금이므로
2. **컨텍스트 윈도우** — 한 번에 넣을 수 있는 대화 분량이 줄어든다
3. **속도** — 출력 토큰을 하나씩 생성하므로

> **참고: 모델을 바꾸면 이 표를 다시 재야 한다.** 다른 모델로 `MODEL`을 바꿔서 위 셀을 다시 실행해보면 숫자가 달라지는 걸 볼 수 있다.

## 3. 응답에서 사용량 읽기

`count_tokens()`는 입력만 셀 수 있다. **실제 비용은 출력 토큰까지 합쳐야** 나오고, 그건 호출 후 응답에 들어있다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
prompt = "파이썬의 리스트와 튜플 차이를 3줄로 설명해줘."

r = client.models.generate_content(model=MODEL, contents=prompt)

print(r.text)
print()
print("--- Gemini 사용량 ---")
print("입력(prompt)     :", r.usage_metadata.prompt_token_count)
print("출력(candidates) :", r.usage_metadata.candidates_token_count)
print("합계(total)      :", r.usage_metadata.total_token_count)
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: prompt / candidates / total 세 값이 나오는지 본다

In [21]:
prompt = "파이썬의 리스트와 튜플 차이를 3줄로 설명해줘."

r = client.models.generate_content(model=MODEL, contents=prompt)

print(r.text)
print()
print("--- Gemini 사용량 ---")
print("입력(prompt)     :", r.usage_metadata.prompt_token_count)
print("출력(candidates) :", r.usage_metadata.candidates_token_count)
print("합계(total)      :", r.usage_metadata.total_token_count)

리스트는 대괄호(`[]`)를 사용하며 생성 후 값을 **변경할 수 있지만**, 튜플은 소괄호(`()`)를 사용하며 생성 후 값을 **변경할 수 없습니다.**

리스트는 데이터의 추가, 수정, 삭제가 빈번한 경우에 유리하고, 튜플은 데이터가 변하지 않도록 고정해야 할 때 사용합니다.

튜플은 리스트보다 메모리 사용량이 적고 처리 속도가 약간 더 빠르다는 장점이 있습니다.

--- Gemini 사용량 ---
입력(prompt)     : 21
출력(candidates) : 106
합계(total)      : 127


**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
oa_r = oa.responses.create(model=OA_MODEL, input=prompt)

print(oa_r.output_text)
print()
print("--- OpenAI 사용량 ---")
print("입력(input)  :", oa_r.usage.input_tokens)
print("출력(output) :", oa_r.usage.output_tokens)
print("합계(total)  :", oa_r.usage.total_tokens)
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 같은 프롬프트인데 Gemini와 토큰 수가 다른지 본다

In [22]:
oa_r = oa.responses.create(model=OA_MODEL, input=prompt)

print(oa_r.output_text)
print()
print("--- OpenAI 사용량 ---")
print("입력(input)  :", oa_r.usage.input_tokens)
print("출력(output) :", oa_r.usage.output_tokens)
print("합계(total)  :", oa_r.usage.total_tokens)

1. 리스트는 가변(mutable) 데이터 구조로, 요소를 추가, 수정, 삭제할 수 있지만, 튜플은 불변(immutable) 데이터 구조로, 생성 후 요소를 변경할 수 없다.
2. 리스트는 대괄호([])로 생성하고, 튜플은 괄호(())로 생성한다.
3. 리스트는 메모리 사용이 클 수 있지만, 튜플은 메모리 소모가 적고, 해시 가능하기 때문에 딕셔너리의 키로 사용할 수 있다.

--- OpenAI 사용량 ---
입력(input)  : 27
출력(output) : 117
합계(total)  : 144


같은 프롬프트인데 토큰 수가 다르다 — **토크나이저가 다르기 때문이다.** 그래서 비용 비교는 토큰 수가 아니라 **최종 금액**으로 해야 한다.

## 4. 단가를 찾아 비용 계산하기

> **주의: 단가는 이 노트북에 적어두지 않았다. 자주 바뀌기 때문이다.**
> 아래 공식 페이지를 직접 열어 `gpt-4o-mini`의 100만 토큰당 단가를 확인하고 TODO를 채운다.
>
> - OpenAI: https://platform.openai.com/docs/pricing
> - Gemini: https://ai.google.dev/pricing
>
> **확인할 때 반드시 볼 것: 입력(input)과 출력(output) 단가가 다르다.**

In [23]:
# TODO: 가격 페이지에서 gpt-4o-mini의 Standard 단가를 찾아 채운다 (단위: USD / 100만 토큰)
#       입력과 출력 단가가 다르다. 헷갈리지 않게 확인할 것.
PRICE_IN_PER_1M = 0.15    # 숫자 형식 예시: 1.23  (실제 값 아님)
PRICE_OUT_PER_1M = 0.60   # 숫자 형식 예시: 4.56  (실제 값 아님)

# TODO: 오늘 환율을 검색해 채운다 (1 USD = ? KRW)
USD_KRW = 1391.24            # 숫자 형식 예시: 1400  (실제 값 아님)


def cost(input_tokens: int, output_tokens: int) -> dict:
    """토큰 수를 받아 비용을 달러와 원으로 계산한다."""
    if None in (PRICE_IN_PER_1M, PRICE_OUT_PER_1M, USD_KRW):
        raise ValueError(
            "위 TODO 3개를 먼저 채운다.\n"
            "  - gpt-4o-mini 입력/출력 단가: https://platform.openai.com/docs/pricing\n"
            "  - 오늘 환율(1 USD = ? KRW)"
        )
    usd = (input_tokens / 1_000_000) * PRICE_IN_PER_1M + (output_tokens / 1_000_000) * PRICE_OUT_PER_1M
    return {"usd": usd, "krw": usd * USD_KRW}


# 3절에서 실제로 한 호출의 비용
c = cost(oa_r.usage.input_tokens, oa_r.usage.output_tokens)
print(f"이 호출 1회 비용: ${c['usd']:.8f}  (약 {c['krw']:.4f}원)")

이 호출 1회 비용: $0.00007425  (약 0.1033원)


### 확인해보기

- 입력 단가와 출력 단가의 **배수**가 얼마인가? 왜 출력이 더 비싼지는 `[배포용] 1_LLM API 동작 원리와 토큰·과금.md`에서 다뤘다.
- 가격 페이지에 `Batch`, `Flex`, `Priority` 같은 다른 요금제도 보일 것이다. **지금은 `Standard` 기준으로 채운다.** (Batch는 즉시 응답이 필요 없는 대량 처리용이라 더 싸다 — 실무 최적화 수단 중 하나다)

숫자가 너무 작아서 실감이 안 날 것이다. **1회 호출은 원 단위도 안 된다.**
그래서 개인이 공부하는 수준에서는 비용이 사실상 문제가 되지 않는다. 문제는 **서비스 규모로 커질 때**다.

## 5. 서비스 규모로 환산

방금 그 호출을 **실제 서비스에서 매일 반복**한다고 가정하고 월 비용을 계산한다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
in_tok = oa_r.usage.input_tokens
out_tok = oa_r.usage.output_tokens

scenarios = [
    ("개인 학습", 1, 20),
    ("사내 소규모 툴", 50, 10),
    ("스타트업 서비스", 1_000, 10),
    ("중견 서비스", 50_000, 10),
]

rows = []
for name, users, calls_per_day in scenarios:
    monthly_calls = users * calls_per_day * 30
    c = cost(in_tok * monthly_calls, out_tok * monthly_calls)
    rows.append({
        "시나리오": name,
        "사용자": f"{users:,}",
        "월 호출수": f"{monthly_calls:,}",
        "월 비용(USD)": f"${c['usd']:,.2f}",
        "월 비용(원)": f"{c['krw']:,.0f}원",
    })

pd.DataFrame(rows)
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 시나리오별 월 비용이 표로 나오는지 본다

In [24]:
in_tok = oa_r.usage.input_tokens
out_tok = oa_r.usage.output_tokens

scenarios = [
    ("개인 학습", 1, 20),
    ("사내 소규모 툴", 50, 10),
    ("스타트업 서비스", 1_000, 10),
    ("중견 서비스", 50_000, 10),
]

rows = []
for name, users, calls_per_day in scenarios:
    monthly_calls = users * calls_per_day * 30
    c = cost(in_tok * monthly_calls, out_tok * monthly_calls)
    rows.append({
        "시나리오": name,
        "사용자": f"{users:,}",
        "월 호출수": f"{monthly_calls:,}",
        "월 비용(USD)": f"${c['usd']:,.2f}",
        "월 비용(원)": f"{c['krw']:,.0f}원",
    })

pd.DataFrame(rows)

,시나리오,사용자,월 호출수,월 비용(USD),월 비용(원)
0,개인 학습,1,600,$0.04,62원
1,사내 소규모 툴,50,"15,000",$1.11,"1,549원"
2,스타트업 서비스,"1,000","300,000",$22.27,"30,990원"
3,중견 서비스,"50,000","15,000,000","$1,113.75","1,549,494원"


### 여기서 얻어야 할 감각

- 호출 1회는 소수점 여섯째 자리지만, **사용자 수 × 호출 수 × 30일이 곱해지면 실제 비용이 된다.**
- 그래서 **프롬프트를 한 줄 줄이는 것**이 서비스 규모에서는 의미 있는 절감이 된다.
- 특히 `system` 지침은 **모든 호출에 매번 붙는다.** 여기가 가장 효과가 큰 최적화 지점이다.

## 6. 무료 한도와 `429` 체험

유료 API는 돈으로 제한하지만, **무료 티어는 횟수와 토큰 수로 제한한다.**

| 약어 | 의미 |
| --- | --- |
| RPM | Requests Per Minute (분당 요청 수) |
| RPD | Requests Per Day (하루 요청 수) |
| TPM | Tokens Per Minute (분당 토큰 수) |

한도를 넘기면 **`429 RESOURCE_EXHAUSTED`** 가 돌아온다. 아래에서 일부러 발생시켜 본다.

> **참고: 에러 메시지 안에 정확한 한도가 들어있다.** 교재를 준비하며 실제로 받은 응답:
> ```
> Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests,
> limit: 15, model: gemini-3.1-flash-lite
> ...
> "retryDelay": "17s"
> ```
> **분당 15회**가 한도이고, **17초 뒤에 재시도하라**는 것까지 알려준다. 문서를 찾아 헤매기 전에 **에러 메시지부터 끝까지 읽는 습관**을 들인다.

> **주의: 이 셀은 본인의 무료 한도를 실제로 소모한다.** 한 번만 실행한다.
> 이 실습 이후 Gemini 호출이 막히면, 1분 정도 기다리면 분당 한도는 회복된다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
# 짧은 요청을 빠르게 반복해서 분당 한도에 도달시켜 본다
hit, kind = None, None

for i in range(1, 26):
    try:
        client.models.generate_content(model=MODEL, contents=f"{i} 다음 숫자만 답해줘")
        print(f"{i:2d}회 성공")
    except Exception as e:
        msg = str(e)
        hit = i
        kind = "429 (내 한도 초과)" if ("429" in msg or "RESOURCE_EXHAUSTED" in msg) else \
               "503 (서버 과부하)" if ("503" in msg or "UNAVAILABLE" in msg) else \
               f"기타 ({type(e).__name__})"
        print(f"{i:2d}회 실패 → {kind}")
        print("   ", msg[:180])
        break

print()
if hit and kind.startswith("429"):
    print(f"{hit}회째에 무료 한도에 걸렸다. 1분쯤 지나면 분당 한도는 회복된다.")
elif hit:
    print(f"{hit}회째에 멈췄지만 한도 초과가 아니다 — {kind}. 내 할당량은 소모되지 않았다.")
else:
    print("25회까지 한도에 걸리지 않았다 (한도가 넉넉한 계정이다).")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 몇 회째에 429가 나는지 본다 (한 번만 실행할 것)

In [25]:
# 짧은 요청을 빠르게 반복해서 분당 한도에 도달시켜 본다
hit, kind = None, None

for i in range(1, 26):
    try:
        client.models.generate_content(model=MODEL, contents=f"{i} 다음 숫자만 답해줘")
        print(f"{i:2d}회 성공")
    except Exception as e:
        msg = str(e)
        hit = i
        kind = "429 (내 한도 초과)" if ("429" in msg or "RESOURCE_EXHAUSTED" in msg) else \
               "503 (서버 과부하)" if ("503" in msg or "UNAVAILABLE" in msg) else \
               f"기타 ({type(e).__name__})"
        print(f"{i:2d}회 실패 → {kind}")
        print("   ", msg[:180])
        break

print()
if hit and kind.startswith("429"):
    print(f"{hit}회째에 무료 한도에 걸렸다. 1분쯤 지나면 분당 한도는 회복된다.")
elif hit:
    print(f"{hit}회째에 멈췄지만 한도 초과가 아니다 — {kind}. 내 할당량은 소모되지 않았다.")
else:
    print("25회까지 한도에 걸리지 않았다 (한도가 넉넉한 계정이다).")

 1회 성공
 2회 성공
 3회 성공
 4회 성공
 5회 성공
 6회 성공
 7회 성공
 8회 성공
 9회 성공
10회 성공
11회 성공
12회 성공
13회 성공
14회 성공
15회 성공
16회 성공
17회 성공
18회 실패 → 429 (내 한도 초과)
    429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to

18회째에 무료 한도에 걸렸다. 1분쯤 지나면 분당 한도는 회복된다.


### 실무에서는 이렇게 처리한다

`429`는 **버그가 아니라 정상적인 응답**이다. 서비스라면 반드시 처리해야 한다.

```python
import time

def call_with_retry(prompt, max_retry=4):
    for attempt in range(max_retry):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt)
        except Exception as e:
            # 429(내 한도 초과)와 503(서버 과부하) 둘 다 일시적이라 재시도 대상이다
            transient = any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE"))
            if not transient or attempt == max_retry - 1:
                raise                        # 키 오류·모델명 오류는 재시도해도 소용없다
            wait = 2 ** attempt              # 1초 → 2초 → 4초 (지수 백오프)
            print(f"일시적 오류, {wait}초 후 재시도")
            time.sleep(wait)
```

노트북 01에서 만든 `gen()` 헬퍼와 같은 구조다.

**지수 백오프(exponential backoff)** — 재시도 간격을 점점 늘린다. 같은 간격으로 재시도하면 한도가 회복되기 전에 또 때려서 상황이 더 나빠진다.

**재시도하면 안 되는 오류를 구분하는 게 핵심이다.** 잘못된 API 키나 없는 모델명은 100번 재시도해도 똑같이 실패한다.

> `3_chat-service` 확장 과정에서는 이 `429`를 잡아 HTTP `503`으로 바꿔 프런트엔드에 내려준다.

In [9]:
import time

def call_with_retry(prompt, max_retry=4):
    for attempt in range(max_retry):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt)
        except Exception as e:
            # 429(내 한도 초과)와 503(서버 과부하) 둘 다 일시적이라 재시도 대상이다
            transient = any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE"))
            if not transient or attempt == max_retry - 1:
                raise                        # 키 오류·모델명 오류는 재시도해도 소용없다
            wait = 2 ** attempt              # 1초 → 2초 → 4초 (지수 백오프)
            print(f"일시적 오류, {wait}초 후 재시도")
            time.sleep(wait)

## 7. 연습문제

### 연습 2-1. 출력 길이를 줄이면 비용이 얼마나 줄어드나

같은 질문을 **길게 답하도록** 한 번, **짧게 답하도록** 한 번 호출해서 비용 차이를 계산한다.
(힌트: `system` 지침으로 "3문장 이내" / "자세히 길게"를 지정한다)

In [ ]:
question = "머신러닝이 무엇인지 설명해줘."

# TODO: 길게 답하는 버전으로 호출 → long_r
# TODO: 짧게 답하는 버전으로 호출 → short_r

# TODO: 각각의 비용을 cost() 로 계산하고, 절감률(%)을 출력한다

In [31]:
question = "머신러닝이 무엇인지 설명해줘."

# TODO: 길게 답하는 버전으로 호출 → long_r
config = types.GenerateContentConfig(system_instruction="너는 천체물리학자다. 전문 용어를 사용해 3문장 이내로 정확하게 설명한다.")
long_r = client.models.generate_content(model=MODEL, contents=question, config=config)
   
# TODO: 짧게 답하는 버전으로 호출 → short_r
config = types.GenerateContentConfig(system_instruction="너는 초등학교 3학년에게 설명하는 선생님이다. 쉬운 낱말만 쓰고 1문장 이내로 답한다.")
short_r = client.models.generate_content(model=MODEL, contents=question, config=config)

# 답변 출력
print("=== 긴 답변 ===")
print(long_r.text)

print("\n=== 짧은 답변 ===")
print(short_r.text)


=== 긴 답변 ===
머신러닝은 데이터 내의 잠재적 패턴과 통계적 상관관계를 알고리즘이 스스로 학습하여, 명시적인 프로그래밍 없이도 미지의 입력값에 대한 예측이나 분류를 수행하는 최적화 기법입니다. 이는 고차원 파라미터 공간에서의 손실 함수를 최소화하는 경사 하강법과 같은 최적화 과정을 통해 가중치를 조정하며 모델의 일반화 성능을 극대화합니다. 결과적으로 천체 관측 데이터에서 은하의 형태를 분류하거나 외계 행성을 탐지하는 것과 같이, 방대한 비정형 데이터로부터 복잡한 물리적 물리적 비선형성을 추론하는 데 핵심적인 역할을 수행합니다.

=== 짧은 답변 ===
머신러닝은 컴퓨터에게 수많은 문제를 보여주고 스스로 공부해서 정답을 찾게 하는 방법이야.


In [28]:
# TODO: 각각의 비용을 cost() 로 계산하고, 절감률(%)을 출력한다
long_r_cost = cost(long_r.usage_metadata.prompt_token_count, long_r.usage_metadata.candidates_token_count)
short_r_cost = cost(short_r.usage_metadata.prompt_token_count, short_r.usage_metadata.candidates_token_count)


rate = short_r_cost['usd']/long_r_cost['usd']*100
print(rate)

21.783876500857634


### 연습 2-2. 프롬프트 다이어트

아래 `system` 지침은 불필요하게 길다. **의미는 유지하면서 토큰 수를 절반 이하로** 줄여보고,
월 100만 호출 서비스에서 얼마나 절감되는지 계산한다.

In [ ]:
verbose_prompt = """당신은 매우 친절하고 상냥하며 사용자를 진심으로 돕고자 하는 마음을 가진
아주 훌륭한 고객 상담 도우미입니다. 사용자가 질문을 하면 언제나 최선을 다해서 성심성의껏
답변을 해주시기 바랍니다. 답변을 할 때에는 반드시 존댓말을 사용해 주시고, 너무 길지 않게
간결하게 답변해 주시면 감사하겠습니다."""

# TODO: 같은 의미로 짧게 다시 쓴다
short_prompt = """"""

# TODO: 두 지침의 토큰 수를 count_tokens 로 비교한다

# TODO: 월 100만 호출 기준으로 절감되는 비용을 계산한다 (지침은 매 호출마다 입력 토큰에 포함된다)

## 정리

- [ ] 토큰이 글자 수와 다르다는 것을 직접 확인했다
- [ ] 한국어가 영어보다 토큰을 더 먹는다는 것을 측정했다
- [ ] 응답에서 입력/출력 토큰을 읽을 수 있다
- [ ] 공식 단가를 찾아 호출 1회의 비용을 계산할 수 있다
- [ ] 서비스 규모에서 비용이 어떻게 커지는지 감이 생겼다
- [ ] `429`를 직접 발생시켜 보고 대응 방법을 안다

**다음** → [03_parameters.ipynb](./03_parameters.ipynb)
이번엔 출력을 **제어**하는 파라미터를 다룬다. 그중 `max_tokens`는 방금 배운 비용과 직결된다.